# Day 03 — Training Paradigms

**Time budget: 60 minutes.** Segment headings carry their own timebox. If you overrun badly, stop
and split the topic across two days rather than rushing the hands-on parts.

## How to use this notebook

1. Read the markdown, then run the code cell under it before reading on. The cells build on each
   other, so run them in order.
2. When a cell says *predict first*, write down your guess before running it. Being wrong is the
   part that sticks.
3. Finish with the exercises at the bottom. Solutions are there, but try first.

## Agenda

| # | Segment | Time |
| - | ------- | ---- |
| 0 | The training problem | 3 min |
| 1 | Supervised learning failure | 8 min |
| 2 | Pretraining: learning language | 12 min |
| 3 | Fine-tuning implementation | 10 min |
| 4 | Scale and data reality | 5 min |
| 5 | RLHF: human alignment | 15 min |
| 6 | The alignment imperative | 7 min |
| 7 | Exercises and quiz | — |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import random

np.random.seed(42)
random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print("Ready to explore training paradigms!")

---
## 0. The Training Problem (3 min)

**The central question:** How do you train a model to be helpful, harmless, and honest?

This seems simple: collect examples of good behavior, train the model to imitate them. But this naive approach fails for several reasons:

1. **Data scarcity:** Getting high-quality examples for every possible scenario is impossible
2. **Generalization:** Models need to handle situations not in training data
3. **Alignment:** "Correct" behavior often depends on human values, not just patterns

**The solution:** A three-stage training pipeline that each solves a different piece of the puzzle.

In [ ]:
# Let's visualize the training paradigm evolution
paradigms = [
    "Direct Supervised\n(Naive)",
    "Pretraining\n(Foundation)", 
    "Fine-tuning\n(Specialization)",
    "RLHF\n(Alignment)"
]

data_sizes = [1000, 1000000000000, 100000, 50000]  # Typical data sizes
success_rates = [0.2, 0.4, 0.8, 0.9]  # Relative success on end task

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Data requirements (log scale)
ax1.bar(range(len(paradigms)), np.log10(data_sizes), color=['red', 'blue', 'green', 'orange'])
ax1.set_xlabel('Training Paradigm')
ax1.set_ylabel('Log10(Data Size)')
ax1.set_title('Data Requirements by Training Stage')
ax1.set_xticks(range(len(paradigms)))
ax1.set_xticklabels(paradigms, rotation=0)

# Add actual numbers as text
for i, size in enumerate(data_sizes):
    if size >= 1e9:
        label = f"{size/1e9:.0f}B"
    elif size >= 1e6:
        label = f"{size/1e6:.0f}M"
    elif size >= 1e3:
        label = f"{size/1e3:.0f}K"
    else:
        label = str(size)
    ax1.text(i, np.log10(data_sizes[i]) + 0.2, label, ha='center', va='bottom')

# Plot 2: Success rates
ax2.plot(range(len(paradigms)), success_rates, 'o-', linewidth=2, markersize=8)
ax2.set_xlabel('Training Evolution')
ax2.set_ylabel('Task Success Rate')
ax2.set_title('Performance Improvement Through Training Stages')
ax2.set_xticks(range(len(paradigms)))
ax2.set_xticklabels([p.replace('\n', ' ') for p in paradigms], rotation=45)
ax2.set_ylim(0, 1)
ax2.grid(True, alpha=0.3)

# Add success rate labels
for i, rate in enumerate(success_rates):
    ax2.text(i, rate + 0.02, f'{rate:.1f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("Key insight: More sophisticated training requires more data but achieves better results.")
print("Each stage solves a different fundamental problem.")

---
## 1. Supervised Learning Failure (8 min)

Let's start by trying the "obvious" approach: directly train a model on the end task we care about. We'll simulate training a chatbot to be helpful by showing it examples of good conversations.

**Prediction:** This will fail due to insufficient data and poor generalization.

In [ ]:
# Simulate a simple "helpful chatbot" dataset
# We'll represent conversations as simple feature vectors

def generate_conversation_features(num_examples, feature_dim=10):
    """
    Generate fake conversation features:
    - Features represent things like: question complexity, politeness, domain, etc.
    - Labels represent: 0 = unhelpful response, 1 = helpful response
    """
    X = np.random.randn(num_examples, feature_dim)
    
    # Create a complex decision boundary (helpful responses need multiple conditions)
    # Helpful if: polite (feature 0 > 0) AND clear question (feature 1 > 0) AND appropriate domain (feature 2 < 1)
    helpful = (X[:, 0] > 0) & (X[:, 1] > 0) & (X[:, 2] < 1)
    
    # Add noise - 10% of labels are flipped
    noise = np.random.random(num_examples) < 0.1
    y = helpful.astype(int)
    y[noise] = 1 - y[noise]
    
    return X, y

# Generate small training set (this is what we'd realistically have for a specific task)
small_X, small_y = generate_conversation_features(500)  # Very limited data
test_X, test_y = generate_conversation_features(1000)   # Test set

print(f"Training examples: {len(small_X)}")
print(f"Test examples: {len(test_X)}")
print(f"Helpful responses in training: {np.mean(small_y)*100:.1f}%")
print(f"Helpful responses in test: {np.mean(test_y)*100:.1f}%")

In [ ]:
# Train a model directly on the end task
direct_model = LogisticRegression(random_state=42)
direct_model.fit(small_X, small_y)

# Evaluate performance
train_pred = direct_model.predict(small_X)
test_pred = direct_model.predict(test_X)

train_acc = accuracy_score(small_y, train_pred)
test_acc = accuracy_score(test_y, test_pred)

print(f"Direct supervised learning results:")
print(f"Training accuracy: {train_acc:.3f}")
print(f"Test accuracy: {test_acc:.3f}")
print(f"Generalization gap: {train_acc - test_acc:.3f}")

# Show confidence on predictions
test_probs = direct_model.predict_proba(test_X)[:, 1]
print(f"\nAverage confidence: {np.mean(test_probs):.3f}")
print(f"Low confidence predictions (< 0.6): {np.mean(test_probs < 0.6)*100:.1f}%")

In [ ]:
# Visualize the learning curve - how performance changes with data
training_sizes = [50, 100, 200, 300, 400, 500]
train_scores = []
test_scores = []

for size in training_sizes:
    # Train on subset of data
    subset_X = small_X[:size]
    subset_y = small_y[:size]
    
    model = LogisticRegression(random_state=42)
    model.fit(subset_X, subset_y)
    
    train_pred = model.predict(subset_X)
    test_pred = model.predict(test_X)
    
    train_scores.append(accuracy_score(subset_y, train_pred))
    test_scores.append(accuracy_score(test_y, test_pred))

# Plot learning curve
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(training_sizes, train_scores, 'o-', label='Training', linewidth=2)
plt.plot(training_sizes, test_scores, 's-', label='Test', linewidth=2)
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.title('Direct Supervised Learning Curve')
plt.legend()
plt.grid(True, alpha=0.3)

# Show the plateau effect
plt.subplot(1, 2, 2)
improvement = np.diff(test_scores)
plt.bar(range(len(improvement)), improvement, alpha=0.7)
plt.xlabel('Additional Data Added')
plt.ylabel('Test Accuracy Improvement')
plt.title('Diminishing Returns from More Data')
plt.xticks(range(len(improvement)), [f'{training_sizes[i]}→{training_sizes[i+1]}' for i in range(len(improvement))])
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"1. Large generalization gap: {train_scores[-1] - test_scores[-1]:.3f}")
print(f"2. Plateau effect: Last 100 examples improved test accuracy by only {improvement[-1]:.3f}")
print(f"3. Low absolute performance: {test_scores[-1]:.3f} accuracy")
print("\nThis is why direct supervised learning fails for complex tasks!")

**The failure is clear:**
1. **Overfitting:** Large gap between train and test performance
2. **Data hunger:** Performance plateaus quickly, more data helps little
3. **Poor generalization:** Model memorizes training examples rather than learning general principles

This is the motivation for **pretraining**: we need the model to learn general language understanding before we specialize it.

---
## 2. Pretraining: Learning Language (12 min)

**The insight:** Before learning to be helpful, the model should learn language itself. Pretraining teaches general language patterns from massive unlabeled text using **self-supervised learning**.

**Key idea:** Predict the next word given previous words. This simple task requires understanding:
- Syntax (grammar rules)
- Semantics (word meanings)
- World knowledge (facts about the world)
- Reasoning (logical relationships)

In [ ]:
# Simulate pretraining with a simple next-word prediction task
# We'll use character-level prediction for simplicity

class SimpleLanguageModel:
    def __init__(self, vocab_size=26):
        self.vocab_size = vocab_size
        self.char_to_idx = {chr(ord('a') + i): i for i in range(26)}
        self.idx_to_char = {i: chr(ord('a') + i) for i in range(26)}
        
        # Simple transition probabilities (like a Markov model)
        self.transitions = np.random.random((vocab_size, vocab_size))
        # Normalize to probabilities
        self.transitions = self.transitions / self.transitions.sum(axis=1, keepdims=True)
        
    def encode_text(self, text):
        """Convert text to indices"""
        text = text.lower().replace(' ', '').replace('.', '').replace(',', '')
        return [self.char_to_idx.get(c, 0) for c in text if c in self.char_to_idx]
    
    def train_on_text(self, text, learning_rate=0.1):
        """Update transition probabilities based on text"""
        indices = self.encode_text(text)
        
        # Count transitions in the text
        for i in range(len(indices) - 1):
            current_char = indices[i]
            next_char = indices[i + 1]
            
            # Increase probability of this transition
            self.transitions[current_char, next_char] += learning_rate
            
        # Renormalize
        self.transitions = self.transitions / self.transitions.sum(axis=1, keepdims=True)
    
    def predict_next(self, context):
        """Predict next character given context"""
        if not context:
            return np.random.choice(self.vocab_size)
        
        last_char_idx = self.char_to_idx.get(context[-1].lower(), 0)
        probs = self.transitions[last_char_idx]
        return np.random.choice(self.vocab_size, p=probs)
    
    def generate_text(self, seed="", length=20):
        """Generate text using the learned model"""
        text = seed.lower()
        
        for _ in range(length):
            next_idx = self.predict_next(text)
            next_char = self.idx_to_char[next_idx]
            text += next_char
        
        return text

# Create model
lm = SimpleLanguageModel()
print("Language model initialized!")

In [ ]:
# Simulate pretraining with a large corpus
pretraining_corpus = [
    "the quick brown fox jumps over the lazy dog",
    "artificial intelligence is transforming the world", 
    "machine learning models learn patterns from data",
    "neural networks are inspired by biological brains",
    "deep learning has revolutionized computer vision",
    "natural language processing helps computers understand text",
    "transformer models use attention mechanisms for processing",
    "pretraining enables transfer learning to new tasks",
    "large language models can generate coherent text",
    "fine tuning adapts pretrained models to specific domains"
] * 100  # Repeat to simulate larger corpus

print(f"Pretraining corpus size: {len(pretraining_corpus)} texts")
print(f"Total characters: {sum(len(text) for text in pretraining_corpus)}")

# Test generation before training
print("\nBefore pretraining:")
print(f"Generated: '{lm.generate_text(seed='the', length=15)}'")

# Train on the corpus
for i, text in enumerate(pretraining_corpus):
    lm.train_on_text(text)
    if (i + 1) % 200 == 0:
        print(f"Processed {i + 1}/{len(pretraining_corpus)} texts...")

# Test generation after training  
print("\nAfter pretraining:")
for seed in ['the', 'neural', 'machine']:
    generated = lm.generate_text(seed=seed, length=15)
    print(f"'{seed}' → '{generated}'")

In [ ]:
# Analyze what the model learned
def analyze_learned_patterns(model, char_pairs):
    """Show transition probabilities for interesting character pairs"""
    print("Learned character transition patterns:")
    for from_char, to_chars in char_pairs.items():
        if from_char in model.char_to_idx:
            from_idx = model.char_to_idx[from_char]
            print(f"\nAfter '{from_char}':")
            
            # Get top transitions
            probs = model.transitions[from_idx]
            top_indices = np.argsort(probs)[-5:][::-1]  # Top 5
            
            for idx in top_indices:
                char = model.idx_to_char[idx]
                prob = probs[idx]
                print(f"  '{char}': {prob:.3f}")

# Analyze common patterns
interesting_patterns = {
    't': ['h', 'e', 'r'],  # 'th', 'te', 'tr' common
    'e': ['r', 'n', 'd'],  # 'er', 'en', 'ed' common
    'n': ['g', 'e', 't'],  # 'ng', 'ne', 'nt' common
}

analyze_learned_patterns(lm, interesting_patterns)

# Show transition matrix visualization
plt.figure(figsize=(10, 8))
plt.imshow(lm.transitions, cmap='Blues', aspect='auto')
plt.colorbar(label='Transition Probability')
plt.xlabel('Next Character')
plt.ylabel('Current Character')
plt.title('Learned Character Transition Probabilities')

# Add character labels (sample every few for readability)
ticks = range(0, 26, 2)
labels = [chr(ord('a') + i) for i in ticks]
plt.xticks(ticks, labels)
plt.yticks(ticks, labels)

plt.tight_layout()
plt.show()

print("\nKey insight: The model learned statistical patterns of English!")
print("Even this simple model captures language structure through self-supervision.")

**What pretraining accomplishes:**
1. **Language structure:** Learns grammar, syntax, common patterns
2. **World knowledge:** Absorbs facts and relationships from text
3. **Generalizable features:** Builds representations useful for many tasks
4. **Scale effects:** More data and parameters → better representations

This creates a **foundation model** with broad language understanding that can be adapted to specific tasks.

---
## 3. Fine-tuning Implementation (10 min)

Now we take our pretrained model and adapt it to the specific task we care about. **Fine-tuning** uses the pretrained representations as a starting point, requiring much less task-specific data.

**Key insight:** The pretrained model has learned general language understanding. We just need to teach it how to apply that knowledge to our specific task.

In [ ]:
# Simulate the benefit of pretraining for our chatbot task
# We'll represent "pretraining" as having learned better feature representations

def simulate_pretrained_features(X):
    """
    Simulate how pretraining would transform raw features into better representations
    In reality, this would be the transformer's hidden states
    """
    # Add learned language features (simulated as combinations of raw features)
    enhanced_features = []
    
    # Original features
    enhanced_features.append(X)
    
    # "Learned" interaction features (what pretraining would discover)
    interaction1 = X[:, [0, 1]].prod(axis=1, keepdims=True)  # Politeness * Clarity
    interaction2 = np.maximum(X[:, [0, 2]], 0).sum(axis=1, keepdims=True)  # Positive sentiment combo
    interaction3 = (X[:, [1, 3, 5]] ** 2).sum(axis=1, keepdims=True)  # Question complexity features
    
    enhanced_features.extend([interaction1, interaction2, interaction3])
    
    # Concatenate all features
    return np.hstack(enhanced_features)

# Compare direct training vs pretrained + fine-tuning
print("Comparison: Direct vs Pretrained+Fine-tuned")
print("=" * 50)

# Method 1: Direct training (what we did before)
direct_model = LogisticRegression(random_state=42)
direct_model.fit(small_X, small_y)
direct_test_acc = accuracy_score(test_y, direct_model.predict(test_X))

print(f"Direct training accuracy: {direct_test_acc:.3f}")

# Method 2: Pretrained features + fine-tuning
pretrained_small_X = simulate_pretrained_features(small_X)
pretrained_test_X = simulate_pretrained_features(test_X)

finetuned_model = LogisticRegression(random_state=42)
finetuned_model.fit(pretrained_small_X, small_y)
finetuned_test_acc = accuracy_score(test_y, finetuned_model.predict(pretrained_test_X))

print(f"Pretrained + fine-tuned accuracy: {finetuned_test_acc:.3f}")
print(f"Improvement: +{finetuned_test_acc - direct_test_acc:.3f}")

print(f"\nFeature dimensions:")
print(f"Original: {small_X.shape[1]}")
print(f"With pretrained representations: {pretrained_small_X.shape[1]}")

In [ ]:
# Show data efficiency of fine-tuning
# How much task-specific data do we need with vs without pretraining?

data_sizes = [50, 100, 150, 200, 300, 500]
direct_scores = []
finetuned_scores = []

for size in data_sizes:
    # Direct training
    subset_X = small_X[:size]
    subset_y = small_y[:size]
    
    direct_model = LogisticRegression(random_state=42, max_iter=1000)
    direct_model.fit(subset_X, subset_y)
    direct_pred = direct_model.predict(test_X)
    direct_scores.append(accuracy_score(test_y, direct_pred))
    
    # Fine-tuned training  
    pretrained_subset_X = simulate_pretrained_features(subset_X)
    
    finetuned_model = LogisticRegression(random_state=42, max_iter=1000)
    finetuned_model.fit(pretrained_subset_X, subset_y)
    finetuned_pred = finetuned_model.predict(pretrained_test_X)
    finetuned_scores.append(accuracy_score(test_y, finetuned_pred))

# Plot comparison
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(data_sizes, direct_scores, 'o-', label='Direct Training', linewidth=2, markersize=6)
plt.plot(data_sizes, finetuned_scores, 's-', label='Pretrained + Fine-tuned', linewidth=2, markersize=6)
plt.xlabel('Task-Specific Training Examples')
plt.ylabel('Test Accuracy')
plt.title('Data Efficiency: Direct vs Fine-tuned')
plt.legend()
plt.grid(True, alpha=0.3)

# Show the advantage
plt.subplot(1, 2, 2)
advantages = np.array(finetuned_scores) - np.array(direct_scores)
plt.bar(range(len(data_sizes)), advantages, alpha=0.7, color='green')
plt.xlabel('Training Set Size')
plt.ylabel('Fine-tuning Advantage')
plt.title('Performance Boost from Pretraining')
plt.xticks(range(len(data_sizes)), data_sizes)
plt.grid(True, alpha=0.3)

# Add value labels
for i, adv in enumerate(advantages):
    plt.text(i, adv + 0.01, f'+{adv:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nFine-tuning advantages:")
print(f"1. Better performance: {max(advantages):.3f} accuracy improvement")
print(f"2. Data efficiency: Achieves {finetuned_scores[2]:.3f} with 150 examples")
print(f"   vs {direct_scores[-1]:.3f} with 500 examples (direct training)")
print(f"3. Faster convergence: Good performance with minimal task data")

# Find data equivalence
target_acc = finetuned_scores[1]  # Accuracy with 100 finetuned examples
direct_needed = None
for i, acc in enumerate(direct_scores):
    if acc >= target_acc:
        direct_needed = data_sizes[i]
        break

if direct_needed:
    print(f"\nData efficiency: 100 fine-tuned examples = {direct_needed} direct examples")
    print(f"Fine-tuning needs {direct_needed/100:.1f}x less task-specific data!")
else:
    print(f"\nDirect training never reaches fine-tuned performance of {target_acc:.3f}")

**Fine-tuning benefits:**
1. **Data efficiency:** Needs 3-10x less task-specific data
2. **Better performance:** Achieves higher accuracy on specialized tasks
3. **Faster training:** Starts from good representations rather than random
4. **Transfer learning:** Knowledge from pretraining transfers to new domains

This is why the pretrain → fine-tune paradigm dominates modern NLP!

---
## 4. Scale and Data Reality (5 min)

Let's put the training paradigms in real-world context. The scale of modern AI training is mind-boggling, and understanding the numbers helps explain why this approach works.

In [ ]:
# Real-world training scales
training_stages = {
    'GPT-3 Pretraining': {
        'data_tokens': 300_000_000_000,  # 300B tokens
        'compute_flops': 3.14e23,        # ~3×10²³ FLOPs
        'cost_estimate': 4_600_000,      # ~$4.6M
        'duration_days': 34,
        'gpus': 10000
    },
    'Fine-tuning (typical)': {
        'data_tokens': 1_000_000,        # 1M tokens
        'compute_flops': 1e18,           # ~10¹⁸ FLOPs  
        'cost_estimate': 1000,           # ~$1K
        'duration_days': 0.5,
        'gpus': 8
    },
    'RLHF (ChatGPT-style)': {
        'data_tokens': 10_000_000,       # 10M tokens (conversations)
        'compute_flops': 1e20,           # ~10²⁰ FLOPs
        'cost_estimate': 200_000,        # ~$200K
        'duration_days': 7,
        'gpus': 1000
    }
}

# Visualize the scales
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
stages = list(training_stages.keys())
colors = ['blue', 'green', 'orange']

# Data scale (log)
data_amounts = [training_stages[stage]['data_tokens'] for stage in stages]
axes[0,0].bar(range(len(stages)), np.log10(data_amounts), color=colors)
axes[0,0].set_title('Training Data Scale (Log10 Tokens)')
axes[0,0].set_xticks(range(len(stages)))
axes[0,0].set_xticklabels(stages, rotation=45, ha='right')
axes[0,0].set_ylabel('Log10(Tokens)')

# Add actual numbers
for i, amount in enumerate(data_amounts):
    if amount >= 1e9:
        label = f"{amount/1e9:.0f}B"
    elif amount >= 1e6:
        label = f"{amount/1e6:.0f}M"
    else:
        label = f"{amount/1e3:.0f}K"
    axes[0,0].text(i, np.log10(amount) + 0.2, label, ha='center', va='bottom')

# Cost scale (log)
costs = [training_stages[stage]['cost_estimate'] for stage in stages]
axes[0,1].bar(range(len(stages)), np.log10(costs), color=colors)
axes[0,1].set_title('Training Cost Scale (Log10 USD)')
axes[0,1].set_xticks(range(len(stages)))
axes[0,1].set_xticklabels(stages, rotation=45, ha='right')
axes[0,1].set_ylabel('Log10(USD)')

for i, cost in enumerate(costs):
    if cost >= 1e6:
        label = f"${cost/1e6:.1f}M"
    elif cost >= 1e3:
        label = f"${cost/1e3:.0f}K"
    else:
        label = f"${cost}"
    axes[0,1].text(i, np.log10(cost) + 0.1, label, ha='center', va='bottom')

# Duration
durations = [training_stages[stage]['duration_days'] for stage in stages]
axes[1,0].bar(range(len(stages)), durations, color=colors)
axes[1,0].set_title('Training Duration (Days)')
axes[1,0].set_xticks(range(len(stages)))
axes[1,0].set_xticklabels(stages, rotation=45, ha='right')
axes[1,0].set_ylabel('Days')

# GPU count (log)
gpu_counts = [training_stages[stage]['gpus'] for stage in stages]
axes[1,1].bar(range(len(stages)), np.log10(gpu_counts), color=colors)
axes[1,1].set_title('GPU Requirements (Log10)')
axes[1,1].set_xticks(range(len(stages)))
axes[1,1].set_xticklabels(stages, rotation=45, ha='right')
axes[1,1].set_ylabel('Log10(GPUs)')

for i, gpu_count in enumerate(gpu_counts):
    axes[1,1].text(i, np.log10(gpu_count) + 0.1, f'{gpu_count}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\nScale insights:")
print(f"1. Pretraining dominates cost: {costs[0]/sum(costs)*100:.1f}% of total training budget")
print(f"2. Data efficiency: Fine-tuning uses {data_amounts[1]/data_amounts[0]*100:.3f}% of pretraining data")
print(f"3. Compute efficiency: Fine-tuning uses ~{training_stages['Fine-tuning (typical)']['compute_flops']/training_stages['GPT-3 Pretraining']['compute_flops']:e} of pretraining compute")
print(f"4. RLHF is expensive: {costs[2]/costs[1]:.0f}x more than standard fine-tuning")

**Scale takeaways:**
1. **Pretraining is expensive:** Most of the cost is in building the foundation model
2. **Fine-tuning is accessible:** 1000x cheaper than pretraining
3. **RLHF adds significant cost:** But essential for deployment
4. **Amortization effect:** One pretrained model serves many fine-tuned applications

This explains why companies like OpenAI can offer API access: the pretraining cost is amortized across millions of users.

---
## 5. RLHF: Human Alignment (15 min)

Fine-tuning creates capable models, but **capability ≠ alignment**. A model might be very good at language tasks but still generate harmful, biased, or unhelpful content.

**RLHF (Reinforcement Learning from Human Feedback)** solves this by training models to optimize for human preferences rather than just predicting text patterns.

**The process:**
1. Generate multiple responses to prompts
2. Humans rank the responses by quality/helpfulness
3. Train a reward model to predict human preferences
4. Use reinforcement learning to maximize reward

In [ ]:
# Simulate the RLHF process with a simple example
# We'll create a scenario where we want helpful, harmless responses

class SimpleRLHF:
    def __init__(self):
        # Simulate different response styles with simple attributes
        # [helpfulness, harmlessness, relevance, politeness]
        self.response_styles = {
            'helpful_good': np.array([0.9, 0.8, 0.9, 0.8]),
            'helpful_rude': np.array([0.8, 0.7, 0.8, 0.2]),
            'unhelpful_safe': np.array([0.3, 0.9, 0.4, 0.9]),
            'harmful': np.array([0.2, 0.1, 0.3, 0.3]),
            'generic': np.array([0.5, 0.6, 0.5, 0.6])
        }
        
        # Human preference weights (what humans actually care about)
        self.human_weights = np.array([0.4, 0.3, 0.2, 0.1])  # Helpfulness > harmlessness > relevance > politeness
        
        # Initialize reward model (starts random)
        self.reward_weights = np.random.randn(4) * 0.1
    
    def human_preference_score(self, response_features):
        """True human preference (hidden during training)"""
        return np.dot(response_features, self.human_weights)
    
    def reward_model_score(self, response_features):
        """Learned reward model prediction"""
        return np.dot(response_features, self.reward_weights)
    
    def collect_human_feedback(self, response_pairs, num_comparisons=50):
        """Simulate collecting human preference data"""
        feedback_data = []
        
        for _ in range(num_comparisons):
            # Sample two random responses
            resp1_name, resp2_name = np.random.choice(list(response_pairs.keys()), 2, replace=False)
            resp1_features = response_pairs[resp1_name]
            resp2_features = response_pairs[resp2_name]
            
            # Human chooses based on true preference
            score1 = self.human_preference_score(resp1_features)
            score2 = self.human_preference_score(resp2_features)
            
            # Add some noise to human judgments
            score1 += np.random.normal(0, 0.1)
            score2 += np.random.normal(0, 0.1)
            
            preferred = 0 if score1 > score2 else 1
            feedback_data.append({
                'response1': resp1_features,
                'response2': resp2_features, 
                'preferred': preferred
            })
        
        return feedback_data
    
    def train_reward_model(self, feedback_data, learning_rate=0.01, epochs=100):
        """Train reward model on human feedback"""
        for epoch in range(epochs):
            total_loss = 0
            
            for comparison in feedback_data:
                resp1_features = comparison['response1']
                resp2_features = comparison['response2']
                preferred = comparison['preferred']
                
                # Predict rewards
                reward1 = self.reward_model_score(resp1_features)
                reward2 = self.reward_model_score(resp2_features)
                
                # Loss: reward model should rank preferred response higher
                if preferred == 0:  # response1 preferred
                    loss = max(0, reward2 - reward1 + 0.1)  # Margin loss
                    if loss > 0:
                        # Update weights to increase reward1, decrease reward2
                        self.reward_weights += learning_rate * resp1_features
                        self.reward_weights -= learning_rate * resp2_features
                else:  # response2 preferred
                    loss = max(0, reward1 - reward2 + 0.1)
                    if loss > 0:
                        self.reward_weights += learning_rate * resp2_features
                        self.reward_weights -= learning_rate * resp1_features
                
                total_loss += loss
            
            if epoch % 20 == 0:
                print(f"Epoch {epoch}: Loss = {total_loss:.3f}")

# Initialize RLHF system
rlhf = SimpleRLHF()
print("RLHF system initialized!")
print(f"\nResponse styles available: {list(rlhf.response_styles.keys())}")
print(f"Human preference weights (true): {rlhf.human_weights}")
print(f"Reward model weights (initial): {rlhf.reward_weights}")

In [ ]:
# Step 1: Collect human feedback on response pairs
print("Step 1: Collecting human preference data...")
feedback_data = rlhf.collect_human_feedback(rlhf.response_styles, num_comparisons=100)

print(f"Collected {len(feedback_data)} preference comparisons")

# Show some examples
print("\nSample human preferences:")
for i in range(3):
    comparison = feedback_data[i]
    resp1_score = rlhf.human_preference_score(comparison['response1'])
    resp2_score = rlhf.human_preference_score(comparison['response2'])
    preferred = "Response 1" if comparison['preferred'] == 0 else "Response 2"
    
    print(f"\nComparison {i+1}:")
    print(f"  Response 1 human score: {resp1_score:.3f}")
    print(f"  Response 2 human score: {resp2_score:.3f}")
    print(f"  Human preferred: {preferred}")

# Step 2: Train reward model
print("\n" + "="*50)
print("Step 2: Training reward model on human feedback...")
rlhf.train_reward_model(feedback_data)

print(f"\nFinal reward model weights: {rlhf.reward_weights}")
print(f"True human weights:        {rlhf.human_weights}")
print(f"Alignment quality:         {np.corrcoef(rlhf.reward_weights, rlhf.human_weights)[0,1]:.3f}")

In [ ]:
# Step 3: Evaluate reward model performance
print("Step 3: Testing reward model accuracy...")

# Test on all response pairs
response_names = list(rlhf.response_styles.keys())
human_scores = []
reward_scores = []

for name in response_names:
    features = rlhf.response_styles[name]
    human_score = rlhf.human_preference_score(features)
    reward_score = rlhf.reward_model_score(features)
    
    human_scores.append(human_score)
    reward_scores.append(reward_score)
    
    print(f"{name:15} | Human: {human_score:.3f} | Reward Model: {reward_score:.3f}")

# Visualize alignment
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(human_scores, reward_scores, s=100, alpha=0.7)
for i, name in enumerate(response_names):
    plt.annotate(name.replace('_', '\n'), (human_scores[i], reward_scores[i]), 
                xytext=(5, 5), textcoords='offset points', fontsize=8)

# Add perfect correlation line
min_score, max_score = min(human_scores + reward_scores), max(human_scores + reward_scores)
plt.plot([min_score, max_score], [min_score, max_score], 'r--', alpha=0.5, label='Perfect alignment')

plt.xlabel('Human Preference Score')
plt.ylabel('Reward Model Score')
plt.title('Reward Model vs Human Preferences')
plt.legend()
plt.grid(True, alpha=0.3)

# Show ranking comparison
plt.subplot(1, 2, 2)
human_ranking = np.argsort(human_scores)[::-1]  # Best to worst
reward_ranking = np.argsort(reward_scores)[::-1]

x = range(len(response_names))
plt.bar(x, [human_ranking[i] for i in range(len(x))], alpha=0.7, label='Human Ranking')
plt.bar([i + 0.3 for i in x], [reward_ranking[i] for i in range(len(x))], alpha=0.7, label='Reward Model Ranking')

plt.xlabel('Response Type (ordered by human preference)')
plt.ylabel('Rank (0=best, 4=worst)')
plt.title('Ranking Comparison')
plt.xticks([i + 0.15 for i in x], [response_names[human_ranking[i]].replace('_', '\n') for i in x], rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate ranking accuracy
ranking_correlation = np.corrcoef(human_ranking, reward_ranking)[0, 1]
print(f"\nRanking correlation: {ranking_correlation:.3f}")
print(f"The reward model {'successfully' if ranking_correlation > 0.7 else 'partially'} learned human preferences!")

In [ ]:
# Step 4: Policy optimization (simplified)
# In real RLHF, this would be PPO (Proximal Policy Optimization)
# We'll simulate by showing how a model would shift its output distribution

print("Step 4: Policy optimization simulation...")

# Simulate model output probabilities before and after RLHF
response_names = list(rlhf.response_styles.keys())

# Before RLHF: uniform distribution (model generates all types equally)
before_probs = np.ones(len(response_names)) / len(response_names)

# After RLHF: probability proportional to reward (simplified)
reward_scores = [rlhf.reward_model_score(rlhf.response_styles[name]) for name in response_names]
# Softmax to convert to probabilities
exp_rewards = np.exp(np.array(reward_scores) * 2)  # Temperature = 0.5
after_probs = exp_rewards / np.sum(exp_rewards)

# Visualize the shift
plt.figure(figsize=(12, 5))

x = range(len(response_names))
width = 0.35

plt.subplot(1, 2, 1)
bars1 = plt.bar([i - width/2 for i in x], before_probs, width, label='Before RLHF', alpha=0.7, color='red')
bars2 = plt.bar([i + width/2 for i in x], after_probs, width, label='After RLHF', alpha=0.7, color='green')

plt.xlabel('Response Type')
plt.ylabel('Generation Probability')
plt.title('Model Output Distribution: Before vs After RLHF')
plt.xticks(x, [name.replace('_', '\n') for name in response_names], rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)

# Add value labels
for bar, prob in zip(bars1, before_probs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{prob:.2f}', ha='center', va='bottom', fontsize=8)
for bar, prob in zip(bars2, after_probs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{prob:.2f}', ha='center', va='bottom', fontsize=8)

# Show expected quality improvement
plt.subplot(1, 2, 2)
human_scores = [rlhf.human_preference_score(rlhf.response_styles[name]) for name in response_names]

expected_quality_before = np.sum(np.array(human_scores) * before_probs)
expected_quality_after = np.sum(np.array(human_scores) * after_probs)

qualities = [expected_quality_before, expected_quality_after]
labels = ['Before RLHF', 'After RLHF']
colors = ['red', 'green']

bars = plt.bar(labels, qualities, color=colors, alpha=0.7)
plt.ylabel('Expected Human Preference Score')
plt.title('Quality Improvement from RLHF')
plt.grid(True, alpha=0.3)

for bar, quality in zip(bars, qualities):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{quality:.3f}', ha='center', va='bottom')

improvement = expected_quality_after - expected_quality_before
plt.text(0.5, max(qualities) * 0.8, f'Improvement:\n+{improvement:.3f}', 
         ha='center', va='center', bbox=dict(boxstyle='round', facecolor='lightblue'))

plt.tight_layout()
plt.show()

print(f"\nRLHF Results:")
print(f"Expected quality before RLHF: {expected_quality_before:.3f}")
print(f"Expected quality after RLHF:  {expected_quality_after:.3f}")
print(f"Quality improvement: +{improvement:.3f} ({improvement/expected_quality_before*100:.1f}%)")

# Show which response types are now favored
print(f"\nMost likely response after RLHF: {response_names[np.argmax(after_probs)]}")
print(f"Probability: {max(after_probs):.2f} (was {1/len(response_names):.2f} before)")

**RLHF accomplishes:**
1. **Preference learning:** Captures human values in a reward model
2. **Policy optimization:** Shifts model behavior toward preferred outputs
3. **Quality improvement:** Measurable increase in human-rated quality
4. **Alignment:** Model behavior better matches human intentions

This is why ChatGPT feels so different from raw GPT-3 - RLHF makes it helpful, harmless, and honest!

---
## 6. The Alignment Imperative (7 min)

**The crucial insight:** Capability without alignment is dangerous. As models become more powerful, ensuring they do what humans actually want becomes increasingly important.

RLHF is just the beginning - alignment remains an active and critical research area.

In [ ]:
# Demonstrate the alignment problem with a simple example
# What happens when a capable model is misaligned?

def simulate_misalignment_scenarios():
    """
    Show how different training objectives can lead to very different behaviors
    even with the same underlying model capability
    """
    
    # Scenario: A model trained to "be helpful" with different interpretations
    scenarios = {
        'Raw Language Model': {
            'objective': 'Predict next token',
            'behavior': 'Mimics training data patterns',
            'problems': ['May repeat biases', 'No safety filtering', 'Unpredictable'],
            'helpfulness': 0.4,
            'safety': 0.3,
            'truthfulness': 0.5
        },
        'Naive Fine-tuning': {
            'objective': 'Maximize user engagement', 
            'behavior': 'Says what users want to hear',
            'problems': ['May agree with harmful requests', 'Tells pleasing lies', 'Sycophantic'],
            'helpfulness': 0.7,
            'safety': 0.4,
            'truthfulness': 0.4
        },
        'Instruction Following': {
            'objective': 'Follow instructions exactly',
            'behavior': 'Literal instruction compliance',
            'problems': ['May follow harmful instructions', 'No judgment', 'Overly literal'],
            'helpfulness': 0.8,
            'safety': 0.5,
            'truthfulness': 0.7
        },
        'RLHF Aligned': {
            'objective': 'Maximize human preference',
            'behavior': 'Helpful, harmless, honest',
            'problems': ['Still imperfect', 'Human feedback quality dependent', 'May be overly cautious'],
            'helpfulness': 0.8,
            'safety': 0.8,
            'truthfulness': 0.8
        }
    }
    
    return scenarios

# Analyze alignment approaches
scenarios = simulate_misalignment_scenarios()

# Visualize the trade-offs
plt.figure(figsize=(14, 5))

scenario_names = list(scenarios.keys())
metrics = ['helpfulness', 'safety', 'truthfulness']
colors = ['blue', 'red', 'green']

# Plot 1: Metric comparison
plt.subplot(1, 2, 1)
x = range(len(scenario_names))
width = 0.25

for i, metric in enumerate(metrics):
    values = [scenarios[name][metric] for name in scenario_names]
    plt.bar([j + i*width for j in x], values, width, label=metric.capitalize(), 
           color=colors[i], alpha=0.7)

plt.xlabel('Training Approach')
plt.ylabel('Score (0-1)')
plt.title('Alignment Quality by Training Approach')
plt.xticks([i + width for i in x], [name.replace(' ', '\n') for name in scenario_names], rotation=0)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)

# Plot 2: Overall alignment score (weighted average)
plt.subplot(1, 2, 2)
weights = [0.3, 0.4, 0.3]  # Safety weighted most heavily
overall_scores = []

for name in scenario_names:
    score = sum(scenarios[name][metric] * weight for metric, weight in zip(metrics, weights))
    overall_scores.append(score)

bars = plt.bar(range(len(scenario_names)), overall_scores, 
               color=['red' if s < 0.6 else 'orange' if s < 0.7 else 'green' for s in overall_scores])
plt.xlabel('Training Approach')
plt.ylabel('Overall Alignment Score')
plt.title('Overall Alignment Quality')
plt.xticks(range(len(scenario_names)), [name.replace(' ', '\n') for name in scenario_names], rotation=0)
plt.ylim(0, 1)

# Add score labels
for bar, score in zip(bars, overall_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{score:.2f}', ha='center', va='bottom')

# Add alignment quality regions
plt.axhline(y=0.6, color='red', linestyle='--', alpha=0.5, label='Misaligned')
plt.axhline(y=0.7, color='orange', linestyle='--', alpha=0.5, label='Partially aligned')
plt.axhline(y=0.8, color='green', linestyle='--', alpha=0.5, label='Well aligned')

plt.tight_layout()
plt.show()

print("\nAlignment Analysis:")
for name, scenario in scenarios.items():
    overall_score = sum(scenario[metric] * weight for metric, weight in zip(metrics, weights))
    print(f"\n{name}: {overall_score:.2f}")
    print(f"  Objective: {scenario['objective']}")
    print(f"  Key problems: {', '.join(scenario['problems'][:2])}")

print(f"\nKey insight: Only RLHF achieves good alignment across all dimensions.")
print(f"But even RLHF isn't perfect - alignment research continues!")

In [ ]:
# Show why alignment becomes more important with scale
model_scales = ['Small (1B)', 'Medium (10B)', 'Large (100B)', 'Very Large (1T)']
capabilities = [0.3, 0.6, 0.8, 0.9]  # Model capabilities increase with scale

# Impact of misalignment scales with capability
misalignment_risk = [cap * 0.3 for cap in capabilities]  # 30% misalignment
aligned_benefit = [cap * 0.9 for cap in capabilities]    # 90% alignment

plt.figure(figsize=(10, 6))

x = range(len(model_scales))
plt.plot(x, capabilities, 'o-', linewidth=3, markersize=8, label='Model Capabilities')
plt.plot(x, aligned_benefit, 's-', linewidth=3, markersize=8, label='Aligned Model Impact', color='green')
plt.plot(x, misalignment_risk, '^-', linewidth=3, markersize=8, label='Misaligned Model Risk', color='red')

plt.fill_between(x, aligned_benefit, capabilities, alpha=0.3, color='green', label='Alignment benefit')
plt.fill_between(x, 0, misalignment_risk, alpha=0.3, color='red', label='Misalignment risk')

plt.xlabel('Model Scale')
plt.ylabel('Impact Magnitude')
plt.title('Why Alignment Becomes Critical at Scale')
plt.xticks(x, model_scales)
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)

# Add annotations
plt.annotate('Gap widens\nwith scale!', xy=(2, 0.5), xytext=(1, 0.7),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, ha='center')

plt.tight_layout()
plt.show()

print("\nAlignment urgency increases with model scale because:")
print(f"1. Capability gap: {aligned_benefit[-1] - misalignment_risk[-1]:.1f} at largest scale")
print(f"2. Risk amplification: Misaligned large models cause more harm")
print(f"3. Deployment stakes: More powerful models need better safety")
print(f"4. Societal impact: Large models affect millions of users")

print(f"\nThis is why AI alignment is a critical research priority!")

**Why alignment matters more as models scale:**
1. **Capability amplifies everything:** Both benefits and risks scale with model power
2. **Deployment consequences:** More capable models have broader impact
3. **Harder to fix later:** Alignment issues are easier to prevent than correct
4. **Societal stakes:** Misaligned AGI could be catastrophic

This is why the AI safety community focuses so much on alignment research - getting it right is essential for beneficial AI.

---
## 7. Exercises

Try each before opening the solution.

**Exercise 1.** Compare data efficiency: Create learning curves showing how much data each training paradigm needs to reach 80% of its final performance. Which is most data-efficient?

**Exercise 2.** Design a reward model: Given features [accuracy, helpfulness, safety, creativity], design human preference weights for a) a chatbot, b) a creative writing assistant, c) a medical AI. How do the weights differ?

**Exercise 3.** Simulate emergence: Create a toy model where capabilities (like in-context learning) only appear above a certain scale threshold. Show how performance changes with model size.

**Exercise 4.** Alignment failure modes: Design scenarios where each training paradigm (pretraining, fine-tuning, RLHF) could fail. What are the specific risks of each approach?

**Exercise 5.** Cost-benefit analysis: Given the training costs from section 4, calculate the break-even point for pretraining vs fine-tuning only. When does pretraining make economic sense?

In [ ]:
# Your scratch space for the exercises.

<details>
<summary><b>Solutions</b> (click to expand)</summary>

```python
# Exercise 1: Data efficiency comparison
def compare_data_efficiency():
    data_points = np.logspace(2, 6, 20)  # 100 to 1M examples
    
    # Different learning rates for each paradigm
    direct_perf = 1 - np.exp(-data_points/100000)  # Slow learning
    pretrain_perf = 1 - np.exp(-data_points/10000)  # Faster learning
    rlhf_perf = 0.8 + 0.2 * (1 - np.exp(-data_points/5000))  # Starts high
    
    # Find 80% performance points
    target = 0.8
    for i, (d, r, p, h) in enumerate(zip(data_points, direct_perf, pretrain_perf, rlhf_perf)):
        if d >= target: print(f"Direct: {data_points[i]:.0f} examples")
        if r >= target: print(f"Pretrain: {data_points[i]:.0f} examples")
        if h >= 0.8*h[-1]: print(f"RLHF: {data_points[i]:.0f} examples")
        break

# Exercise 2: Domain-specific reward models
reward_weights = {
    'chatbot': [0.2, 0.4, 0.3, 0.1],      # Safety > helpfulness > accuracy > creativity
    'creative': [0.1, 0.3, 0.2, 0.4],     # Creativity > helpfulness > safety > accuracy  
    'medical': [0.5, 0.3, 0.2, 0.0]       # Accuracy > helpfulness > safety, no creativity
}

# Exercise 3: Emergence simulation
def simulate_emergence(model_sizes):
    capabilities = []
    for size in model_sizes:
        if size < 1e9:  # Below 1B parameters
            cap = 0.1 * np.log(size/1e6)  # Slow scaling
        else:  # Above 1B - emergence!
            cap = 0.5 + 0.4 * np.log(size/1e9)  # Sudden jump + continued scaling
        capabilities.append(max(0, min(1, cap)))
    return capabilities

# Exercise 4: Failure modes
failure_modes = {
    'pretraining': ['Amplifies training data biases', 'No safety filtering', 'Learns harmful patterns'],
    'fine_tuning': ['Overfits to narrow tasks', 'Loses general capabilities', 'May ignore safety'],
    'rlhf': ['Goodhart\'s law - optimizes proxy metric', 'Human feedback quality issues', 'Reward hacking']
}

# Exercise 5: Economic break-even
def calculate_breakeven(pretrain_cost=5e6, finetune_cost=1e3, applications=1000):
    # Cost per application: pretraining amortized over N apps + fine-tuning each
    cost_with_pretrain = pretrain_cost/applications + finetune_cost
    cost_without_pretrain = finetune_cost * 5  # Assume 5x more data needed
    
    breakeven_apps = pretrain_cost / (cost_without_pretrain - finetune_cost)
    return breakeven_apps
```

</details>

---
## Self-check quiz

If you cannot answer these without scrolling up, reread the segment named in the answer.

1. **Why does direct supervised learning fail for complex AI tasks?**
2. **What does pretraining teach that makes fine-tuning so effective?**
3. **How does RLHF differ from standard supervised fine-tuning?**
4. **Why is the pretraining → fine-tuning → RLHF order important?**
5. **Why does alignment become more critical as models scale up?**

<details>
<summary><b>Answers</b></summary>

1. **Direct supervised learning fails due to data scarcity, poor generalization, and inability to capture the full complexity of human values and preferences. Models overfit to limited examples rather than learning general principles.** (segment 1)

2. **Pretraining teaches general language understanding - syntax, semantics, world knowledge, and reasoning patterns. This creates rich, transferable representations that need much less task-specific data to adapt.** (segment 2)

3. **RLHF optimizes for human preferences rather than imitating training examples. It uses human feedback to learn a reward model, then uses reinforcement learning to maximize that reward rather than just predicting text.** (segment 5)

4. **Each stage builds on the previous: pretraining creates foundational capabilities, fine-tuning specializes them for tasks, and RLHF aligns them with human values. You need capabilities before you can align them properly.** (segments 2-5)

5. **As capabilities increase, both the potential benefits and risks scale proportionally. Misaligned powerful models can cause much more harm, while aligned ones provide much more benefit. The stakes get higher with scale.** (segment 6)

</details>

---
## Where to go next

**Next concept: Emergent Capabilities and Scaling Laws**

You now understand how modern AI systems are trained, but a crucial question remains: **why does this work so well?** 

The next deep dive should explore how capabilities emerge from scale - why larger models with more data suddenly develop abilities like few-shot learning, chain-of-thought reasoning, and instruction following that smaller models lack entirely.

Understanding scaling laws and emergence explains why the training paradigms you just learned are so powerful, and helps predict what capabilities might appear next as models continue to scale up.